## Gerar Matrizes de Performances P e de Meta-características X

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from pymfe.mfe import MFE

warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASETS_DIR = os.path.join(PROJECT_ROOT, "source", "datasets")
SOURCE_DIR = os.path.join(PROJECT_ROOT, "source")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

N_DATASETS = 30          
N_SPLITS = 10            # repetições de hold-out por dataset -> 10 valores pareados por fold
TEST_SIZE = 0.2          # proporção de teste em cada repetição
SEED = 42                # semente para reprodutibilidade
PYMFE_GROUPS = "all"     # extrai meta-características de todas as categorias do pymfe
PYMFE_SUMMARY = ("mean", "sd")  # como resumir medidas que geram múltiplos valores

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASETS_DIR:", DATASETS_DIR)

PROJECT_ROOT: d:\Projetos\DisciplinaMtL
DATASETS_DIR: d:\Projetos\DisciplinaMtL\source\datasets


In [ ]:
def load_dataset(name: str):
    path = os.path.join(DATASETS_DIR, f"{name}.csv")
    df = pd.read_csv(path)
    y = df["target"].values
    X = df.drop(columns=["target"]).values
    return X, y


def make_models():
    return {
        "DT": Pipeline([
            ("clf", DecisionTreeClassifier(random_state=SEED)),
        ]),
        "RF": Pipeline([
            ("clf", RandomForestClassifier(
                n_estimators=100, random_state=SEED, n_jobs=-1)),
        ]),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)),
        ]),
        "NB": Pipeline([
            ("clf", GaussianNB()),
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5)),
        ]),
    }

In [3]:
raw_dataset_names = sorted(f[:-4] for f in os.listdir(DATASETS_DIR) if f.endswith(".csv"))

meta_rows = []
skipped_datasets = []

t_start = time.time()
for name in raw_dataset_names:
    X, y = load_dataset(name)
    t0 = time.time()
    try:
        mfe = MFE(groups=PYMFE_GROUPS, summary=PYMFE_SUMMARY)
        mfe.fit(X, y)
        feat_names, feat_values = mfe.extract()
    except Exception as e:
        skipped_datasets.append((name, repr(e)))
        print(f"{name:35s} FALHOU  ({e!r})")
        continue

    row = {"Dataset": name}
    row.update(dict(zip(feat_names, feat_values)))
    meta_rows.append(row)
    print(f"{name:35s} OK      ({len(feat_names)} meta-características, "
          f"{time.time() - t0:5.1f}s)")

print(f"\nTempo total: {(time.time() - t_start) / 60:.1f} min")
if skipped_datasets:
    print(f"\nDatasets ignorados (falha na extração via pymfe): "
          f"{[n for n, _ in skipped_datasets]}")

matrizX_full = pd.DataFrame(meta_rows).set_index("Dataset")

cols_before = matrizX_full.shape[1]
matrizX_full = matrizX_full.dropna(axis=1, how="any")
print(f"\nColunas com algum valor nulo removidas: {cols_before - matrizX_full.shape[1]} "
      f"(de {cols_before} para {matrizX_full.shape[1]})")

matrizx_path = os.path.join(SOURCE_DIR, "MatrizX.csv")
matrizX_full.to_csv(matrizx_path, index_label="Dataset")
print(f"Salvo: {matrizx_path}  "
      f"({matrizX_full.shape[0]} datasets x {matrizX_full.shape[1]} meta-características)")
matrizX_full.head(5)

annealing                           OK      (179 meta-características,   3.3s)
balloons                            OK      (179 meta-características,   0.1s)
bank                                OK      (179 meta-características,  19.0s)
blood                               OK      (179 meta-características,   0.7s)
car                                 OK      (179 meta-características,   2.3s)
contrac                             OK      (179 meta-características,   2.1s)
dermatology                         OK      (179 meta-características,   1.5s)
echocardiogram                      OK      (179 meta-características,   0.7s)
ecoli                               OK      (179 meta-características,   0.8s)
fertility                           OK      (179 meta-características,   0.6s)
flags                               OK      (179 meta-características,   1.3s)
glass                               OK      (179 meta-características,   0.8s)
hepatitis                           OK      (179 met

,attr_conc.mean,attr_conc.sd,attr_ent.mean,attr_ent.sd,attr_to_inst,c1,c2,can_cor.mean,cat_to_num,ch,...,two_itemset.sd,var.mean,var.sd,var_importance.mean,var_importance.sd,vdb,vdu,w_lambda,wg_dist.mean,wg_dist.sd
Dataset,,,,,,,,,,,,,,,,,,,,,
annealing,3.775931e-01,2.793622e-01,1.060526,0.668992,0.034521,0.512434,0.638352,0.628187,0.0,0.442007,...,0.304720,17951.845436,8.635682e+04,0.032258,0.062473,15.498981,0.000000e+00,0.045675,1.622904,2.129213e-01
balloons,-4.000000e-08,6.911696e-24,1.000000,0.000000,0.250000,0.988699,0.030769,0.755929,0.0,2.333333,...,0.000000,1.000000,0.000000e+00,0.250000,0.105588,2.393416,7.936508e-03,0.428571,1.072501,2.866584e-16
bank,9.080164e-02,1.944305e-01,1.985222,1.378263,0.003539,0.515522,0.743846,0.479993,0.0,78.847906,...,0.218927,1.000000,1.817200e-06,0.062500,0.068910,4.779118,6.981111e-09,0.769606,1.420668,1.379916e-01
blood,2.220620e-01,3.660123e-01,2.882541,0.268578,0.005348,0.791645,0.430934,0.359244,0.0,34.174220,...,0.067623,1.000000,7.212370e-07,0.250000,0.130028,3.337901,0.000000e+00,0.870944,0.332966,1.190131e-01
car,-4.966668e-08,8.332187e-09,1.792481,0.227325,0.003472,0.602870,0.506593,0.358848,0.0,61.345087,...,0.022922,0.999998,1.524163e-06,0.166667,0.091639,3.119716,2.309067e-07,0.460337,1.090737,5.412979e-02


In [ ]:
# datasets_with_meta_features = set(pd.read_csv(matrizx_path)["Dataset"])

# usable_names = sorted(set(raw_dataset_names) & datasets_with_meta_features)
# print(f"Datasets brutos disponíveis: {len(raw_dataset_names)}")
# print(f"Datasets com meta-características extraídas (MatrizX.csv): {len(datasets_with_meta_features)}")
# print(f"Interseção utilizável (com dados brutos E meta-características): {len(usable_names)}")

# dataset_names = usable_names[:N_DATASETS]
# print(f"\nUsando os {N_DATASETS} primeiros datasets desse universo utilizável:")
# for i, n in enumerate(dataset_names, 1):
#     print(f"{i:2d}. {n}")

## Validação cruzada

In [5]:
def run_dataset(name: str):
    '''Roda a validação cruzada (10x hold-out estratificado 80/20) para um
    dataset e retorna uma lista de tuplas (dataset, algoritmo, fold, accuracy).
    '''
    X, y = load_dataset(name)
    splitter = StratifiedShuffleSplit(
        n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=SEED)

    rows = []
    for fold_i, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for algo_name, pipeline in make_models().items():
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            rows.append((name, algo_name, fold_i, acc))

    return rows


all_rows = []
t_start = time.time()
for i, name in enumerate(dataset_names, 1):
    t0 = time.time()
    rows = run_dataset(name)
    all_rows.extend(rows)
    print(f"[{i:2d}/{len(dataset_names)}] {name:32s} "
          f"({time.time() - t0:5.1f}s)")

print(f"\nTempo total: {(time.time() - t_start) / 60:.1f} min")
print(f"Total de resultados (dataset x algoritmo x fold): {len(all_rows)}")

[ 1/30] annealing                        (  5.7s)
[ 2/30] balloons                         (  1.8s)
[ 3/30] bank                             (  5.4s)
[ 4/30] blood                            (  2.4s)
[ 5/30] car                              (  2.7s)
[ 6/30] contrac                          (  2.9s)
[ 7/30] dermatology                      (  2.0s)
[ 8/30] echocardiogram                   (  1.8s)
[ 9/30] ecoli                            (  1.9s)
[10/30] fertility                        (  1.7s)
[11/30] flags                            (  1.8s)
[12/30] glass                            (  1.8s)
[13/30] hepatitis                        (  1.7s)
[14/30] ionosphere                       (  1.8s)
[15/30] iris                             (  1.8s)
[16/30] lenses                           (  1.7s)
[17/30] libras                           (  2.2s)
[18/30] lymphography                     (  1.8s)
[19/30] mammographic                     (  2.1s)
[20/30] nursery                          ( 12.7s)


In [6]:

folds_df = pd.DataFrame(all_rows, columns=["dataset", "algoritmo", "fold", "accuracy"])
folds_path = os.path.join(RESULTS_DIR, "MatrizP_folds_30.csv")
folds_df.to_csv(folds_path, index=False)
print(f"Salvo: {folds_path}  ({len(folds_df)} linhas)")
folds_df.head(10)


Salvo: d:\Projetos\DisciplinaMtL\results\MatrizP_folds_30.csv  (1500 linhas)


,dataset,algoritmo,fold,accuracy
0,annealing,DT,1,0.961111
1,annealing,RF,1,0.950000
2,annealing,SVM,1,0.883333
3,annealing,NB,1,0.561111
4,annealing,KNN,1,0.894444
5,annealing,DT,2,0.983333
6,annealing,RF,2,0.966667
7,annealing,SVM,2,0.855556
8,annealing,NB,2,0.500000
9,annealing,KNN,2,0.883333


In [ ]:
# Matriz P (formato largo): média por fold, uma coluna por algoritmo
P = folds_df.pivot_table(index="dataset", columns="algoritmo",
                          values="accuracy", aggfunc="mean")

# Mantém a ordem alfabética original dos datasets (não a ordem alfabética de pivot_table, que já é a mesma aqui, mas deixei explícito)
P = P.loc[dataset_names]

algo_order = ["DT", "RF", "SVM", "NB", "KNN"]
P = P[algo_order]
P["Best"] = P[algo_order].idxmax(axis=1)

P_path = os.path.join(RESULTS_DIR, "MatrizP_30.csv")
P.to_csv(P_path, index_label="Dataset")
print(f"Salvo: {P_path}")
P


Salvo: d:\Projetos\DisciplinaMtL\results\MatrizP_30.csv


algoritmo,DT,RF,SVM,NB,KNN,Best
dataset,,,,,,
annealing,0.953889,0.958333,0.852778,0.538889,0.878889,RF
balloons,0.600000,0.625000,0.650000,0.525000,0.650000,SVM
bank,0.863094,0.900221,0.894144,0.816796,0.892486,RF
blood,0.719333,0.739333,0.762667,0.734667,0.752667,SVM
car,0.980347,0.984393,0.968497,0.691908,0.935549,RF
contrac,0.495593,0.525763,0.549492,0.467119,0.478644,SVM
dermatology,0.956757,0.974324,0.979730,0.870270,0.971622,SVM
echocardiogram,0.744444,0.844444,0.837037,0.874074,0.814815,NB
ecoli,0.795588,0.894118,0.888235,0.780882,0.873529,RF


In [9]:
print(f"\nDatasets: {P.shape[0]}  |  Algoritmos: {len(algo_order)}")
print("\nVitórias por algoritmo (quantos datasets cada um venceu, na média):")
print(P["Best"].value_counts())


Datasets: 30  |  Algoritmos: 5

Vitórias por algoritmo (quantos datasets cada um venceu, na média):
Best
SVM    13
RF     12
DT      3
NB      1
KNN     1
Name: count, dtype: int64
